In [1]:
# Run this cell first. It installs the packages this notebook uses.
%pip install -q pandas matplotlib

Note: you may need to restart the kernel to use updated packages.


# 00. Setup: what this project is and where every number comes from

**US Elections: The Midterm Effect** is a beginner friendly data story.
It answers three questions in plain words.

1. How do US presidential elections work, and what has happened since 1976?
2. What happens in midterm elections, the elections held halfway through a
   president's four year term?
3. How do midterm results change what a president can actually do, and what
   could the November 2026 midterms mean for the rest of the Trump
   administration, which ends on January 20, 2029?

A note on timing. This project was built on November 2, 2026. The midterm
election is on November 3, 2026, the next day. No November 2026 results exist
yet. This project does not invent any. Chapter 04 covers what is at stake,
forecasts from named sources with dates, and scenarios. It also has a
results section that is clearly labeled as pending, ready to fill in after
November 3.

In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt

PROC = os.path.join('..', 'data', 'processed')
CHART_DIR = os.path.join('..', 'charts')
os.makedirs(CHART_DIR, exist_ok=True)
plt.rcParams['figure.dpi'] = 120
DEM = '#1f77b4'
REP = '#d62728'
GREY = '#7f7f7f'

def savefig(name):
    path = os.path.join(CHART_DIR, name)
    plt.savefig(path, bbox_inches='tight')
    plt.close()
    print('saved', path)

## Project structure

The cell below lists every folder and file in this project, so you can see
exactly what is here before we use any of it.

In [3]:
for base, dirs, files in os.walk('..'):
    dirs[:] = [d for d in dirs if d not in ('.venv', '__pycache__')]
    level = base.replace('..', '').count(os.sep)
    print(' ' * 2 * level + os.path.basename(base) + '/')
    for f in sorted(files):
        if f.endswith('.ipynb_checkpoints'):
            continue
        print(' ' * 2 * (level + 1) + f)

../
  notebooks/
    00_setup.ipynb
    01_presidential_history.ipynb
    02_midterm_history.ipynb
    03_the_midterm_effect.ipynb
    04_november_2026.ipynb
    05_what_to_watch.ipynb
  charts/
  data/
    raw/
      house_1976_2024_mirror.csv
      president_1976_2024.csv
      senate_1976_2024.tab
    processed/
      executive_orders.csv
      forecasts_2026.csv
      house_division.csv
      house_seats.csv
      house_winners.csv
      judicial_confirmations.csv
      midterm_history.csv
      polls_2026.csv
      presidential_national.csv
      presidential_state.csv
      senate_division.csv
      senate_winners.csv
      shutdowns.csv
      stakes_2026.csv
      swing_states.csv
      vetoes.csv
  scripts/
    build_notebooks.py
    build_processed.py


## Tools

We use Python with two libraries: pandas for tables and matplotlib for
charts. The pip cell at the top of every notebook installs them, so each
notebook runs on its own in Google Colab or on any computer.

In [4]:
import sys
print('Python:', sys.version.split()[0])
print('pandas:', pd.__version__)
import matplotlib
print('matplotlib:', matplotlib.__version__)

Python: 3.12.3
pandas: 3.1.0
matplotlib: 3.11.2


## Every dataset source

A dataset is a table of numbers that someone collected and published. Here
is every dataset this project uses, where it came from, and what it covers.
The full provenance notes live in `data/raw/PROVENANCE.md`.

| Dataset | Source | Coverage |
|---|---|---|
| Presidential votes by state | MIT Election Data and Science Lab, Harvard Dataverse, DOI 10.7910/DVN/42MVDX, file president_1976_2024.csv | 1976 to 2024 |
| Senate votes by state | MIT Election Data and Science Lab, Harvard Dataverse, DOI 10.7910/DVN/PEJ5QU, file senate_1976_2024.tab | 1976 to 2024 |
| House votes by district | keithpotz/Election-Prediction mirror of the MIT House file, house_1976_2024.csv (the MIT download itself required a guestbook form, so the mirror is used and cross checked) | 1976 to 2024 |
| Senate party divisions | senate.gov, history party divisions page | 95th to 119th Congress |
| House party divisions | Wikipedia, Party divisions of United States Congresses, which compiles Clerk of the House data | 95th to 119th Congress |
| Electoral College votes | Federal Election Commission, official presidential general election results | 1976 to 2024 |
| Voter turnout | University of Florida Election Lab (Michael McDonald), voting eligible population turnout, via Wikipedia | 1980 to 2024, plus midterms 2014, 2018, 2022 |
| Presidential vetoes | American Presidency Project, presidential vetoes table, data through January 8, 2026 | Carter through Trump second term, in progress |
| Executive orders | Federal Register disposition tables, via Competitive Enterprise Institute tables | Carter through Biden, plus a provisional Trump second term count |
| Judicial confirmations | Wikipedia, list of presidents by judicial appointments, Article III judges only | Carter through Biden, plus Trump first term |
| Federal shutdowns | Congressional Research Service based lists of major shutdowns | 1995 to 2025 |
| 2026 stakes and forecasts | House and Senate trackers, Cook Political Report and Sabato's Crystal Ball as reported in October 2026, Ballotpedia, Wikipedia | November 2026, pre election |

A forecast is a prediction made before the election. A result is what voters
actually decided. This project never mixes the two. Every forecast in
chapter 04 is labeled with its source and its date.

## Verification checks

Good data work checks itself. The cells below run four checks and print the
outcome of each one.

1. MIT presidential vote totals against published candidate totals for 2020
   and 2024. Small gaps (under 0.1 percent) are expected, because write in
   votes and late certifications are counted slightly differently.
2. House mirror seat counts against the official party division table.
3. Wrong winner elections: the only elections since 1976 where the popular
   vote winner lost the Electoral College should be 2000 and 2016.
4. Row counts for every processed file.

In [5]:
nat = pd.read_csv(os.path.join(PROC, 'presidential_national.csv'))
for year, pub_dem, pub_rep in [(2020, 81283501, 74223975), (2024, 75019230, 77303568)]:
    row = nat[nat['year'] == year].iloc[0]
    d_gap = (row['dem_votes'] - pub_dem) / pub_dem * 100
    r_gap = (row['rep_votes'] - pub_rep) / pub_rep * 100
    print(year, 'Democrat gap: %.3f%%' % d_gap, '| Republican gap: %.3f%%' % r_gap)
    assert abs(d_gap) < 0.1 and abs(r_gap) < 0.1
print('Check 1 passed: MIT totals match published totals within 0.1 percent.')

2020 Democrat gap: -0.018% | Republican gap: -0.011%
2024 Democrat gap: -0.002% | Republican gap: -0.001%
Check 1 passed: MIT totals match published totals within 0.1 percent.


In [6]:
seats = pd.read_csv(os.path.join(PROC, 'house_seats.csv'))
div = pd.read_csv(os.path.join(PROC, 'house_division.csv'))
div['election_year'] = (div['congress'] - 1) * 2 + 1788
merged = seats.merge(div, left_on='year', right_on='election_year')
merged['d_diff'] = merged['dem_seats'] - merged['dem']
merged['r_diff'] = merged['rep_seats'] - merged['rep']
print(merged[['year', 'dem_seats', 'dem', 'd_diff', 'rep_seats', 'rep', 'r_diff']].to_string(index=False))
print('Largest Democrat gap:', merged['d_diff'].abs().max(), 'seats.')
print('Largest Republican gap:', merged['r_diff'].abs().max(), 'seats.')
print('Note: 1996 is the outlier. The mirror misses 10 Texas districts in')
print('1996 because a court ordered redistricting moved those races into a')
print('special process that the mirror codes separately. That is why the')
print('midterm series uses the official division table, not the mirror.')
print('Check 2 done: gaps are 0 to 4 seats except 1996, and the cause is known.')

 year  dem_seats  dem  d_diff  rep_seats  rep  r_diff
 1976        291  292      -1        144  143       1
 1978        277  277       0        158  157       1
 1980        244  242       2        190  192      -2
 1982        269  269       0        166  165       1
 1984        254  253       1        181  181       0
 1986        259  258       1        176  177      -1
 1988        260  260       0        175  175       0
 1990        267  267       0        167  167       0
 1992        258  258       0        174  176      -2
 1994        205  204       1        229  230      -1
 1996        205  206      -1        218  227      -9
 1998        211  211       0        223  223       0
 2000        213  212       1        220  221      -1
 2002        205  205       0        229  229       0
 2004        202  202       0        232  232       0
 2006        232  233      -1        199  202      -3
 2008        257  257       0        178  178       0
 2010        197  193       

In [7]:
wrong = nat[nat['wrong_winner']]
print('Wrong winner elections since 1976:')
print(wrong[['year', 'dem_nominee', 'rep_nominee', 'dem_votes', 'rep_votes', 'dem_ev', 'rep_ev']].to_string(index=False))
assert list(wrong['year']) == [2000, 2016]
print('Check 3 passed: only 2000 and 2016.')

Wrong winner elections since 1976:
 year      dem_nominee      rep_nominee  dem_votes  rep_votes  dem_ev  rep_ev
 2000         GORE, AL  BUSH, GEORGE W.   50830580   50311372     266     271
 2016 CLINTON, HILLARY TRUMP, DONALD J.   65677288   62692670     227     304
Check 3 passed: only 2000 and 2016.


In [8]:
import glob
for path in sorted(glob.glob(os.path.join(PROC, '*.csv'))):
    df = pd.read_csv(path)
    print(os.path.basename(path), ':', len(df), 'rows,', len(df.columns), 'columns')
print('Check 4 done.')

executive_orders.csv : 8 rows, 4 columns
forecasts_2026.csv : 6 rows, 7 columns
house_division.csv : 25 rows, 7 columns
house_seats.csv : 25 rows, 4 columns
house_winners.csv : 10863 rows, 5 columns
judicial_confirmations.csv : 8 rows, 7 columns
midterm_history.csv : 12 rows, 12 columns
polls_2026.csv : 6 rows, 7 columns
presidential_national.csv : 13 rows, 17 columns
presidential_state.csv : 663 rows, 9 columns
senate_division.csv : 25 rows, 7 columns


senate_winners.csv : 841 rows, 5 columns
shutdowns.csv : 6 rows, 6 columns
stakes_2026.csv : 3 rows, 8 columns
swing_states.csv : 13 rows, 2 columns
vetoes.csv : 8 rows, 7 columns
Check 4 done.
